### Overview

- Rag Evaluation workflow consists of three main steps

1. Creating a dataset with questions and their expected answers
2. Running your RAG applications on those questions
3. Using Evaluators to measure how well your application performed, looking at the factors 

- Answer  Relevance
- Answer Accuracy
- Retrieval Quality



In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langsmith import Client
client = Client(api_key = os.getenv("langsmithAPI"))

In [22]:


# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['9ae1a9a7-2a45-4bf5-93c7-31e57d048c4e',
  '90f33906-9379-4d97-a001-06842ce61ca1',
  'e30b0581-d91e-4cf0-85b8-d6a8db2ae4ad',
  '96cb63d3-26a5-4921-bbd1-e7b8267cb0a6',
  '77d10779-60fd-4d04-8abe-e95a94110031'],
 'count': 5}

### Define Metrices (LLM As a Judge)

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model = "gpt-4o-mini",
    api_key=  os.getenv("openAIAPI")
)

In [28]:
import os
import openai
from langsmith import wrappers

openai_client = wrappers.wrap_openai(openai.OpenAI(api_key=os.getenv("openAIAPI")))
eval_instructions = "You are an expert professor specialized in grading students answer to questions."


def correctness(input : dict, outputs : dict, reference_outputs: dict) -> bool :
    user_content = f""" 
    You are grading the following question :
    {input['question']}
    Here is the real answer :
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with Correct or Incorrect :
    Grade :
    """

    response = openai_client.chat.completions.create(
        model = "gpt-4o-mini", temperature = 0,
        messages= [
            {"role" : "system", "content" : eval_instructions},
            {"role" : "user", "content" : user_content}
        ]
    ).choices[0].message.content

    return response == "Correct"

    

In [29]:
def concision(output : dict, reference_outputs: dict) -> bool :
    return int(len(output["response"]) < 2* len(reference_outputs['answer']))

### Run Evaluation

In [30]:
default_instructions = "Respond to the user question in a short, concise manner (one short sentence)"

def my_app(question : str, model:  str = "gpt-4o-mini", instructions : str = default_instructions) -> str:
    return openai_client.chat.completions.create(
        model= model,
        temperature =0,
        messages = [
            {"role" : "system", "content" : instructions},
            {"role" : "user ", "content" : question}
        ]
    ).choices[0].message.content

In [31]:
### call my_app for every datapoint
def ls_target(inputs : str) -> dict :
    return {"response" : my_app(inputs['question='])}

In [ ]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-4o-mini-chatbot"
)

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

## Initialise a text splitter with specified chunk size and overlap

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 250, chunk_overlap = 0
)

doc_splits = text_splitter.split_documents(docs_list)

